# Laboratory 17 — Quantum statistics: Bose–Einstein and Fermi–Dirac gases

In this laboratory you add one rule to the course — how many identical particles may share a
single-particle state — and follow what it does. You count the states of two particles under three
rules; rebuild the Bose–Einstein and Fermi–Dirac occupations from explicit sums and draw them at
random; count the states of a box one by one; solve for the chemical potential that holds a gas at
fixed density, and watch it climb as the gas cools; follow a Bose gas through condensation, both in
the ideal limit and in a real box of up to a million atoms; fill a Fermi sea and measure its heat
capacity; rebuild module 16's Planck spectrum from a boson occupation; and finally measure the
conduction electrons of silver in real data, and test the ideal Bose gas against liquid helium.

The route: the objects (Part 1); counting (Part 2); one state and its fluctuations (Part 3); the
states of a box (Part 4); the chemical potential at fixed density (Part 5); Bose–Einstein
condensation (Part 6); the Fermi sea (Part 7); Planck's law as a Bose gas (Part 8); the electrons in
silver (Part 9); liquid helium (Part 10); the automated checks (Part 11); and four explorers
(Part 12).

Work through it in order. Where the notebook asks you to predict, write your prediction in the
cell provided **before** running the next cell. A prediction you have committed to is the only
reliable way to discover that you were wrong.

**Units.** SI throughout: energies in joules (electronvolts only in printed values), densities in
$\mathrm{m^{-3}}$, temperatures in kelvin, heat capacities in $\mathrm{J\,K^{-1}}$ — per mole when
the number of particles is $N_A$.

## Model specification

| | |
|---|---|
| **System** | an ideal gas of identical particles — bosons, fermions, or classical particles — in a rigid cubic box of volume $V$, each standing wave $g_s$-fold degenerate in spin |
| **Dynamics** | none — every curve is thermal equilibrium; the only random numbers are drawn on purpose, through a passed generator |
| **Boundary** | every single-particle state exchanges energy and particles with a reservoir at $(T, \mu)$; $\mu$ is solved for so that the gas holds $N$ particles |
| **Ensemble** | grand canonical, state by state |
| **Ignored** | every interaction between the particles; the shape of a real trap; relativity; for electrons, the lattice beyond a free-electron box |
| **Valid when** | the box's level spacing is far below $\kB T$ (and $\varepsilon_F$); $N \gg 1$ for condensation |
| **Failure modes** | liquid helium-4 (strong interactions); real metals (band structure); a few hundred atoms (a rounded crossover); white dwarfs near the Chandrasekhar mass (relativistic electrons) |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from thermolab import chemical, quantum, radiation
from thermolab.constants import K_B, N_A
from thermolab.partition import thermal_wavelength
from thermolab.validation import relative_error, scaling_exponent

H = radiation.PLANCK_H
EV = 1.602176634e-19  # J

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    DATA = Path("..", "..", "..", "data")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSVs co-located here.
    DATA = Path("data")

# The two gases this laboratory keeps returning to.
M_HE, N_HE = quantum.M_HELIUM4, 146.1 / quantum.M_HELIUM4   # liquid helium-4 at its lambda point
M_E, N_CU = quantum.M_ELECTRON, 8960.0 / 63.546e-3 * N_A    # copper's conduction electrons
T_C = quantum.bec_tc(N_HE, M_HE)
T_F = float(quantum.fermi_temperature(N_CU, M_E))
print(f"helium-4 at 146.1 kg/m^3: n = {N_HE:.4e} m^-3, ideal-gas T_c = {T_C:.3f} K")
print(f"copper's electrons: n = {N_CU:.4e} m^-3, "
      f"eps_F = {quantum.fermi_energy(N_CU, M_E) / EV:.3f} eV, T_F = {T_F:.0f} K")

## Part 1 — The objects

| function | what it gives |
|---|---|
| `count_states(M, N, s)`, `enumerate_states(M, N, s)` | the number, and the list, of states of $N$ particles in $M$ single-particle states, for `s` = `"mb"`, `"be"` or `"fd"` |
| `bose_einstein`, `fermi_dirac`, `maxwell_boltzmann`, `occupation(eps, mu, T, s)` | the mean occupation of a state of energy `eps` |
| `occupation_variance`, `explicit_occupation`, `sample_occupation` | its variance; mean and variance from the explicit grand sum; random draws from its distribution |
| `box_level_unit`, `count_box_states`, `smooth_state_count`, `density_of_states_3d` | the states of a cube, one by one and smoothed; $g(\varepsilon)$ |
| `degeneracy_parameter(n, T, m)`, `occupation_integral(eta, s)`, `solve_eta`, `solve_mu` | $n\lambda^3$; $F(\eta)$; the $\mu$ that holds the density |
| `ideal_quantum_gas(n, T, m, s, g_s)` | $\mu$, energy, pressure, heat capacity and condensate fraction of a gas |
| `bec_tc`, `condensate_fraction`, `finite_box_condensate_fraction` | condensation, ideal and in a box of $N$ atoms |
| `fermi_energy`, `fermi_temperature`, `fermi_velocity`, `fermi_gas_pressure`, `electronic_heat_capacity`, `free_electron_gamma` | the Fermi sea |
| `fit_metal_heat_capacity` | the straight line $C/T = \gamma + \beta T^2$, returning `.gamma`, `.gamma_sigma`, `.beta`, `.debye_temperature` |

### Predict

Commit to an answer for each before running anything.

1. Two identical bosons in two states: what is the chance that they are in the same state?
2. As a gas is cooled at fixed density, which way does $\mu$ move — for bosons, for fermions?
3. At half the condensation temperature, what fraction of an ideal Bose gas is in the lowest state?
4. Silver's electronic coefficient $\gamma$: within what factor of the free-electron prediction?

**Your predictions:**

1.
2.
3.
4.

## Part 2 — Counting

List the states of two particles in two single-particle states under each rule, and the chance of
finding both in one state if every state is equally likely. Then compare the quantum counts of
three particles in $M$ states with the corrected classical count $M^3/3!$ of module 12.

**Your prediction:** for three particles in $1000$ states, how close are the Bose and Fermi counts
to $M^3/3!$?

*(write here before running the next cell)*

In [ ]:
together = {}
for statistics, name in (("mb", "classical (labelled)"), ("be", "bosons"), ("fd", "fermions")):
    states = quantum.enumerate_states(2, 2, statistics)
    together[statistics] = float(np.mean(states.max(axis=1) == 2))
    print(f"{name:21s} {len(states)} states {[tuple(int(k) for k in s) for s in states]}, "
          f"P(both in one state) = {together[statistics]:.3f}")

print("\nthree particles in M states, divided by the corrected classical count M^3 / 3!:")
for levels in (3, 10, 100, 1000):
    classical = levels**3 / 6
    print(f"M = {levels:5d}: bosons {quantum.count_states(levels, 3, 'be') / classical:.4f}, "
          f"fermions {quantum.count_states(levels, 3, 'fd') / classical:.4f}")

The counts $4$, $3$, $1$ are the whole quantum input. Bosons are found together more often than
classical particles and fermions never, with no force anywhere. And when the states far outnumber
the particles both quantum counts become $M^N/N!$: module 12's division by $N!$ was quantum
statistics in its dilute limit.

## Part 3 — One state, and its fluctuations

Each closed form of step 2 is checked against the explicit sum of the state's grand-canonical
weights — $n = 0, 1, 2, \ldots$ for bosons, $0$ or $1$ for fermions, with $1/n!$ for classical
particles. Then occupations are drawn at random from each distribution, with a seeded generator.

**Your prediction:** which statistics has the largest variance at the same mean occupation?

*(write here before running the next cell)*

In [ ]:
T = 300.0
print("   x     statistics   closed form      explicit sum     variance (closed)  variance (sum)")
worst = 0.0
for x in (0.1, 1.0, 3.0):
    for statistics in ("be", "mb", "fd"):
        eps = x * K_B * T
        mean, variance = quantum.explicit_occupation(eps, 0.0, T, statistics)
        closed = quantum.occupation(eps, 0.0, T, statistics)
        closed_variance = quantum.occupation_variance(eps, 0.0, T, statistics)
        worst = max(worst, relative_error(mean, closed), relative_error(variance, closed_variance))
        print(f"{x:5.1f}   {statistics:>6s}      {closed:.12f}   {mean:.12f}   "
              f"{closed_variance:.12f}   {variance:.12f}")
print(f"largest relative difference: {worst:.1e}")

rng = np.random.default_rng(17)
eps = 0.5 * K_B * T
print("\n100 000 draws of one state's occupation at (eps - mu) / kT = 0.5:")
for statistics in ("be", "mb", "fd"):
    draws = quantum.sample_occupation(eps, 0.0, T, statistics, 100_000, rng)
    print(f"  {statistics}: mean {draws.mean():.4f} (predicted "
          f"{quantum.occupation(eps, 0.0, T, statistics):.4f}), variance {draws.var():.4f} "
          f"(predicted {quantum.occupation_variance(eps, 0.0, T, statistics):.4f})")

x = np.linspace(0.02, 6.0, 300)
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
for statistics, colour, label in (("be", "#2563eb", "Bose-Einstein"),
                                  ("mb", "#111827", "Maxwell-Boltzmann"),
                                  ("fd", "#dc2626", "Fermi-Dirac")):
    n = quantum.occupation(x * K_B * T, 0.0, T, statistics)
    left.plot(x, n, color=colour, label=label)
    right.semilogy(x, n, color=colour, label=label)
left.set_ylim(0, 3)
for ax in (left, right):
    ax.set_xlabel("(eps - mu) / k_B T")
    ax.set_ylabel("<n>")
    ax.legend()
plt.tight_layout()
plt.show()

The closed forms are the sums, to rounding. The variances, $n(1 + n)$, $n$ and $n(1 - n)$, put the
three statistics in the same order as the counting did: bosons bunch, fermions avoid. On the
logarithmic plot the three occupations become parallel lines once $(\varepsilon - \mu)/\kB T$ is a
few units: the dilute limit.

## Part 4 — The states of a box

A particle's standing wave in a cube of side $L$ has $\mathbf{k} = (\pi/L)(n_x, n_y, n_z)$ and energy
$\varepsilon_1 |\mathbf{n}|^2$, $\varepsilon_1 = \hbar^2\pi^2/2mL^2$. `count_box_states` counts the
integer triples one by one; the derivation replaced them by the volume of an octant.

In [ ]:
side = 1e-6  # m
unit = quantum.box_level_unit(side, M_HE)
shortfalls = {}
print(" radius |n|   explicit count   smooth count   shortfall   shortfall x |n|")
for radius in (5, 10, 20, 40, 80, 160, 320):
    counted = quantum.count_box_states(side, M_HE, unit * radius**2)
    smooth = quantum.smooth_state_count(side, M_HE, unit * radius**2)
    shortfalls[radius] = 1 - counted / smooth
    print(f"{radius:10d} {counted:16d} {smooth:14.0f}   {100 * shortfalls[radius]:7.3f} %   "
          f"{shortfalls[radius] * radius:6.3f}")
print(f"\nhelium in a 1 cm box: eps_1 / k_B = {quantum.box_level_unit(0.01, M_HE) / K_B:.1e} K")

The shortfall times $|\mathbf{n}|$ settles at $9/4$: the smooth count overcounts only the states on
the octant's faces, exactly as for module 16's cavity. In any box you could build, the level
spacing is a few times $10^{-15}\ \mathrm{K}$, and the smooth density of states is exact for
practical purposes.

## Part 5 — The chemical potential at fixed density

At the density of liquid helium, solve for $\mu$ at each temperature for bosons, classical
particles and (hypothetical, spinless) fermions, and check the classical solver against module 13's
closed form $\mu = \kB T\ln(n/n_Q)$.

**Your prediction:** sketch $\mu/\kB T_c$ against $T/T_c$ for the three gases before running.

*(write here before running the next cell)*

In [ ]:
ratios = np.geomspace(0.3, 30.0, 80)  # T / T_c
mu_curves = {s: np.array([quantum.solve_mu(N_HE, r * T_C, M_HE, s) for r in ratios]) / (K_B * T_C)
             for s in ("mb", "be", "fd")}
module_13 = np.array([chemical.ideal_gas_mu(N_HE, r * T_C, M_HE) for r in ratios]) / (K_B * T_C)
mb_gap = float(np.max(np.abs(mu_curves["mb"] - module_13) / ratios))
print(f"classical solver against module 13's k_B T ln(n / n_Q): largest difference "
      f"{mb_gap:.1e} k_B T")
eps_f_spinless = quantum.fermi_energy(N_HE, M_HE, 1) / (K_B * T_C)
print(f"spinless fermions at this density: eps_F = {eps_f_spinless:.3f} k_B T_c")
print(f"the classical mu crosses zero at T / T_c = {quantum.ZETA_3_2 ** (2 / 3):.4f}")

print("\nn lambda^3   mu/kT: classical   bosons   fermions")
for d in (0.01, 0.1, 1.0, quantum.ZETA_3_2, 10.0):
    print(f"{d:10.3f}   {quantum.solve_eta(d, 'mb'):+16.3f}   {quantum.solve_eta(d, 'be'):+6.3f}   "
          f"{quantum.solve_eta(d, 'fd'):+8.3f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
for statistics, colour, label in (("be", "#2563eb", "bosons"), ("mb", "#111827", "classical"),
                                  ("fd", "#dc2626", "fermions (spinless)")):
    ax.plot(ratios, mu_curves[statistics], color=colour, label=label)
ax.axhline(eps_f_spinless, color="#dc2626", ls=":", lw=1)
ax.axhline(0.0, color="grey", lw=1)
ax.axvline(1.0, color="#2563eb", ls=":", lw=1)
ax.set_xscale("log")
ax.set_ylim(-20, 4)
ax.set_xlabel("T / T_c")
ax.set_ylabel("mu / k_B T_c")
ax.legend()
plt.tight_layout()
plt.show()

All three climb as the gas is cooled. The classical $\mu$ passes zero at $1.90\,T_c$, where it starts
to predict more than one particle in the lowest states — the classical theory leaving its domain.
The bosons' $\mu$ reaches zero at $T_c$ and stays there. The fermions' climbs to $\varepsilon_F$.

## Part 6 — Bose–Einstein condensation

First, the capacity of the excited states: the occupation integral at $\mu = 0$, by quadrature. Then
the condensate fraction — ideal, and computed level by level for a hard-walled box of $N$ atoms.
Then the pressure below $T_c$.

**Your prediction:** in a box of $10^4$ atoms, is $N_0/N$ at $T_c/2$ above or below the ideal
$1 - 0.5^{3/2} = 0.646$?

*(write here before running the next cell)*

In [ ]:
capacity = quantum.occupation_integral(0.0, "be")
print(f"F_BE(0) by quadrature: {capacity:.16f};  zeta(3/2) = {quantum.ZETA_3_2:.16f}")
for n_points in (16, 24, 32, 64):
    error = relative_error(quantum.occupation_integral(0.0, "be", 1.5, n_points), quantum.ZETA_3_2)
    print(f"  {n_points:3d} trapezoid intervals: relative error {error:.1e}")

t = np.linspace(0.0, 1.5, 151)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t, quantum.condensate_fraction(t, 1.0), color="k", lw=2, label="ideal, 1 - (T/T_c)^(3/2)")
for n_atoms, colour in ((1000, "#94a3b8"), (10_000, "#2563eb"), (100_000, "#d97706")):
    ax.plot(t, quantum.finite_box_condensate_fraction(n_atoms, t), ls="--", color=colour,
            label=f"box of {n_atoms:,} atoms")
ax.set_xlabel("T / T_c")
ax.set_ylabel("N_0 / N")
ax.legend()
plt.tight_layout()
plt.show()

window = np.linspace(0.2, 0.8, 7)
exponents, excess = {}, {}
print("\n    atoms   N0/N at T_c/2   excess over 0.646   fitted exponent of 1 - N0/N")
for n_atoms in (1000, 10_000, 100_000, 1_000_000):
    half = float(quantum.finite_box_condensate_fraction(n_atoms, 0.5))
    excess[n_atoms] = half - (1 - 0.5**1.5)
    fraction = np.asarray(quantum.finite_box_condensate_fraction(n_atoms, window))
    exponents[n_atoms] = scaling_exponent(window, 1 - fraction)
    print(f"{n_atoms:9d}   {half:13.4f}   {excess[n_atoms]:17.4f}   {exponents[n_atoms]:12.3f}")

print("\npressure at T = T_c / 2, for three densities:")
pressures = []
for factor in (1, 2, 5):
    gas = quantum.ideal_quantum_gas(factor * N_HE, 0.5 * T_C, M_HE, "be")
    pressures.append(gas.pressure)
    print(f"  {factor} x n: N0/N = {gas.condensate_fraction:.3f}, P = {gas.pressure:.5e} Pa")
at_tc = quantum.ideal_quantum_gas(N_HE, T_C, M_HE, "be")
print(f"at T_c: P / (n k_B T) = {at_tc.pressure / at_tc.classical_pressure:.4f}, "
      f"C / (N k_B) = {at_tc.heat_capacity_per_particle / K_B:.4f}")

The quadrature finds $\zeta(3/2)$ without being told it, with $32$ intervals. A finite box condenses
*early*: its hard walls remove the states on the octant's faces, so the excited states fill sooner.
The excess shrinks slowly as the box grows, and the fitted exponent drifts down toward $3/2$. Below
$T_c$ the pressure is the same for every density: the extra atoms join the condensate, which pushes
on nothing, while the excited atoms keep moving.

## Part 7 — The Fermi sea

Electrons at copper's density, with $g_s = 2$: the closed-form Fermi energy and degeneracy pressure;
then the numerically solved gas at temperatures from $0.001\,T_F$ to $3\,T_F$.

**Your prediction:** at $T = 0.01\,T_F$, what fraction of the classical $\tfrac32\kB$ per electron is
the heat capacity?

*(write here before running the next cell)*

In [ ]:
eps_f = float(quantum.fermi_energy(N_CU, M_E))
print(f"copper: eps_F = {eps_f / EV:.3f} eV, T_F = {T_F:.0f} K, "
      f"v_F = {quantum.fermi_velocity(N_CU, M_E):.3e} m/s")
print(f"degeneracy pressure (2/5) n eps_F = {quantum.fermi_gas_pressure(N_CU, M_E):.3e} Pa; "
      f"classical n k_B T at 300 K = {N_CU * K_B * 300:.3e} Pa")
print(f"n lambda^3 at 300 K = {quantum.degeneracy_parameter(N_CU, 300.0, M_E):.0f}; "
      f"T / T_F = {300 / T_F:.4f}")

cold = quantum.ideal_quantum_gas(N_CU, 1e-3 * T_F, M_E, "fd", 2)
print(f"\nat T = 0.001 T_F: mu / eps_F = {cold.mu / eps_f:.8f}, "
      f"U / (N eps_F) = {cold.energy_per_particle / eps_f:.8f} (3/5 = 0.6)")

fermi_ratios = np.geomspace(1e-3, 3.0, 60)
fermi_gases = [quantum.ideal_quantum_gas(N_CU, r * T_F, M_E, "fd", 2) for r in fermi_ratios]
heat = np.array([g.heat_capacity_per_particle for g in fermi_gases]) / K_B
sommerfeld_ratio = heat[0] / (np.pi**2 / 2 * fermi_ratios[0])
print(f"C / (N k_B) at T = 0.001 T_F is {sommerfeld_ratio:.6f} times (pi^2/2) T/T_F")

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.semilogx(fermi_ratios, [g.mu / eps_f for g in fermi_gases], color="#dc2626")
left.axhline(1.0, color="grey", ls=":")
left.set_xlabel("T / T_F")
left.set_ylabel("mu / eps_F")
right.loglog(fermi_ratios, heat, color="#2563eb", label="ideal Fermi gas")
right.loglog(fermi_ratios, np.pi**2 / 2 * fermi_ratios, "--", color="#dc2626",
             label="(pi^2/2) T/T_F")
right.axhline(1.5, color="k", ls=":", label="classical 3/2")
right.set_ylim(1e-3, 5)
right.set_xlabel("T / T_F")
right.set_ylabel("C_V / (N k_B)")
right.legend()
plt.tight_layout()
plt.show()

At absolute zero the electrons keep a mean energy of $\tfrac35\varepsilon_F$, over four electronvolts,
and a pressure a hundred times the classical one at room temperature. The heat capacity is linear in
$T$ with the coefficient $\pi^2/2$ until about $0.05\,T_F$ — for copper, $4000\ \mathrm{K}$, above its
melting point.

## Part 8 — Planck's law as a Bose gas

Module 16 built Planck's spectrum from the oscillator. Rebuild it here from module 17's occupation:
photons are bosons with $\mu = 0$, so a mode of frequency $\nu$ holds `bose_einstein(h nu, 0, T)`
photons of energy $h\nu$, and there are `radiation.mode_density(nu)` modes per unit volume per
hertz.

In [ ]:
planck_gap = 0.0
for temperature in (2.725, 300.0, 5772.0):
    nu = np.linspace(1e-3, 30.0, 3000) * K_B * temperature / H
    rebuilt = radiation.mode_density(nu) * H * nu * quantum.bose_einstein(H * nu, 0.0, temperature)
    gap = float(np.max(np.abs(rebuilt / radiation.planck_u_nu(nu, temperature) - 1.0)))
    planck_gap = max(planck_gap, gap)
    print(f"T = {temperature:7.3f} K: largest relative difference from planck_u_nu {gap:.1e}")

nu = np.linspace(1e12, 1.5e15, 600)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(nu / 1e12, radiation.planck_u_nu(nu, 5772.0), color="#2563eb", lw=4, alpha=0.4,
        label="module 16: planck_u_nu")
ax.plot(nu / 1e12, radiation.mode_density(nu) * H * nu * quantum.bose_einstein(H * nu, 0.0, 5772.0),
        color="k", lw=1, label="module 17: modes x h nu x n_BE(mu = 0)")
ax.set_xlabel("frequency (THz)")
ax.set_ylabel("u_nu (J s m^-3)")
ax.legend()
plt.tight_layout()
plt.show()

One occupation function, two chapters: the two computations agree to the last digit of a double.

## Part 9 — The electrons in silver

The data file is the one module 16 used: silver's heat capacity at constant pressure from $1$ to
$300\ \mathrm{K}$, tabulated by NIST (Smith and Fickett, 1995), per kilogram. Below a few kelvin
$C = \gamma T + \beta T^3$; this time the electrons' $\gamma$ is the measurement, and the free-electron
gas makes a prediction for it from silver's density alone.

**Your prediction:** below what temperature do silver's electrons carry more of its heat capacity
than its lattice does?

*(write here before running the next cell)*

In [ ]:
MOLAR_MASS, DENSITY = 0.1078682, 10490.0  # kg/mol, kg/m^3
t_ag, cp_per_kg = np.genfromtxt(DATA / "16-silver-heat-capacity.csv", delimiter=",",
                                comments="#").T
c_ag = cp_per_kg * MOLAR_MASS  # J/(mol K)

fits = {}
for t_max in (4.0, 6.0, 8.0):
    window = t_ag <= t_max
    fits[t_max] = quantum.fit_metal_heat_capacity(t_ag[window], c_ag[window])
    print(f"T <= {t_max:.0f} K: gamma = {fits[t_max].gamma * 1e3:.4f} +- "
          f"{fits[t_max].gamma_sigma * 1e3:.4f} mJ/(mol K^2), "
          f"theta_D = {fits[t_max].debye_temperature:.1f} K")
fit = fits[6.0]

n_ag = DENSITY / MOLAR_MASS * N_A
gamma_free = quantum.free_electron_gamma(n_ag)
print(f"\none electron per atom: n = {n_ag:.4e} m^-3, "
      f"eps_F = {quantum.fermi_energy(n_ag, M_E) / EV:.3f} eV, "
      f"T_F = {quantum.fermi_temperature(n_ag, M_E):.0f} K")
print(f"free-electron gamma = {gamma_free * 1e3:.4f} mJ/(mol K^2); measured / free = "
      f"{fit.gamma / gamma_free:.3f}")
crossover = float(np.sqrt(fit.gamma / fit.beta))
share = fit.gamma * t_ag / c_ag
print(f"electrons and lattice are equal at {crossover:.2f} K; electrons' share at 1 K "
      f"{share[0]:.2f}, at 300 K {share[-1]:.4f}")

# Measured gammas from Kittel's table (8th ed., p. 139), against the free-electron values
# computed from each metal's room-temperature density and molar mass.
KITTEL = {"Ag": (10490.0, 107.8682e-3, 1, 0.646e-3), "Au": (19300.0, 196.96657e-3, 1, 0.729e-3),
          "Cu": (8960.0, 63.546e-3, 1, 0.695e-3), "Al": (2700.0, 26.9815385e-3, 3, 1.35e-3)}
print("\nmetal   measured   free electron   ratio")
for metal, (rho, molar, z, measured) in KITTEL.items():
    free = quantum.free_electron_gamma(z * rho / molar * N_A, z)
    print(f"{metal:5s}   {measured * 1e3:8.3f}   {free * 1e3:13.3f}   {measured / free:5.2f}")

low = t_ag <= 6.0
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
grid = np.linspace(0, 37, 50)
left.plot(t_ag[low] ** 2, c_ag[low] / t_ag[low] * 1e3, "o", color="k", label="silver")
left.plot(grid, (fit.gamma + fit.beta * grid) * 1e3, color="#2563eb", label="fit")
left.plot([0], [gamma_free * 1e3], "D", color="#dc2626", label="free-electron gamma")
left.set_xlabel("T^2 (K^2)")
left.set_ylabel("C / T (mJ mol^-1 K^-2)")
left.legend()
right.semilogx(t_ag, share, "o", ms=3, color="k")
right.set_xlabel("T (K)")
right.set_ylabel("gamma T / C")
plt.tight_layout()
plt.show()

The free-electron gas, with no adjustable parameter, predicts silver's $\gamma$ to about $1\%$. The
electrons carry most of silver's heat capacity below $2\ \mathrm{K}$, and under $1\%$ of it at room
temperature: the "missing" heat capacity of the puzzle, measured. Other metals are less free —
copper's ratio is $1.38$ — and the ratio measures how the lattice reshapes the states at the top of
the Fermi sea.

## Part 10 — Liquid helium

The ideal Bose gas at liquid helium's density condenses at $T_c$. The liquid becomes a superfluid
at the lambda point, $2.1768\ \mathrm{K}$ (Donnelly and Barenghi, 1998).

In [ ]:
T_LAMBDA = 2.1768  # K, measured
print(f"ideal-gas T_c = {T_C:.3f} K; lambda point {T_LAMBDA} K; ratio {T_C / T_LAMBDA:.3f}")
print(f"spacing n^(-1/3) = {N_HE ** (-1 / 3) * 1e9:.3f} nm; thermal wavelength at the lambda point "
      f"= {thermal_wavelength(T_LAMBDA, M_HE) * 1e9:.3f} nm")
print(f"n lambda^3 at the lambda point = {quantum.degeneracy_parameter(N_HE, T_LAMBDA, M_HE):.2f}; "
      f"ideal condensate fraction there = {quantum.condensate_fraction(T_LAMBDA, T_C):.3f}")

The ideal gas misses the lambda point by $45\%$ — the right order of magnitude from the mass and the
density alone, for a liquid whose atoms sit closer together than their thermal wavelength and repel
strongly. The model locates the phenomenon; it does not describe the liquid.

### Measurement

Quote two results, each with its uncertainty and the assumptions it rests on:

- $\gamma$(silver) from Part 9, against the free-electron prediction — and say how much it moves if
  the window ends at $4\ \mathrm{K}$ instead of $6$.
- The exponent of $1 - N_0/N$ for a box of $10^6$ atoms from Part 6, against the ideal $3/2$ — and
  say what the trend over $N$ suggests.

## Part 11 — The automated checks

In [ ]:
# 1. The closed forms are the explicit grand sums.
assert worst < 1e-12

# 2. Counting: 4, 3, 1 states; bosons bunch, fermions avoid.
assert [quantum.count_states(2, 2, s) for s in ("mb", "be", "fd")] == [4, 3, 1]
assert together["be"] > together["mb"] > together["fd"] == 0.0

# 3. The smooth density of states misses only the octant's faces.
assert 0 < shortfalls[320] < shortfalls[80] < shortfalls[20]
assert abs(shortfalls[320] * 320 - 9 / 4) < 0.1

# 4. The classical solver is module 13's formula.
assert mb_gap < 1e-12

# 5. Condensation: zeta(3/2) by quadrature; finite boxes approach the ideal from above.
assert relative_error(capacity, quantum.ZETA_3_2) < 1e-14
assert all(e > 0 for e in excess.values())
assert exponents[1000] > exponents[1_000_000] > 1.5 and exponents[1_000_000] < 1.6
assert max(pressures) / min(pressures) - 1 < 1e-12

# 6. The Fermi sea: mu -> eps_F, U/N -> (3/5) eps_F, C linear with pi^2/2.
assert abs(cold.mu / eps_f - 1) < 1e-5 and abs(cold.energy_per_particle / eps_f - 0.6) < 1e-5
assert abs(sommerfeld_ratio - 1) < 1e-4

# 7. Planck's law is the Bose-Einstein distribution at mu = 0.
assert planck_gap < 1e-12

# 8. Silver's electrons are nearly free; the ideal Bose gas misses helium's lambda point.
assert 0.6e-3 < fit.gamma < 0.7e-3 and abs(fit.gamma / gamma_free - 1) < 0.03
assert 1.5 < crossover < 2.5
assert 3.1 < T_C < 3.2
print("all checks passed")

## Part 12 — Explore it yourself

Four explorers; set the controls and press **Run Interact**.

**The occupation explorer.** The three occupations against energy, for the temperature and the
chemical potential you choose (both in the same arbitrary energy unit). Things to try: $\mu$ well
below zero, on the logarithmic axis; then raise $\mu$ toward zero; then try to push the bosons' $\mu$
above zero; then lower the temperature with $\mu$ positive and watch the Fermi function sharpen.

In [ ]:
import ipywidgets as widgets


def occupations(temperature=1.0, mu=-1.0, log_axis=False):
    eps = np.linspace(0.0, 8.0, 801)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    # temperature / K_B makes k_B T equal to `temperature`, in the same unit as eps and mu.
    for statistics, colour, label in (("be", "#2563eb", "Bose-Einstein"),
                                      ("mb", "#111827", "Maxwell-Boltzmann"),
                                      ("fd", "#dc2626", "Fermi-Dirac")):
        if statistics == "be" and mu >= 0.0:
            print("No Bose-Einstein curve: mu must lie below the lowest energy, 0, or the lowest "
                  "state's grand sum diverges.")
            continue
        ax.plot(eps, quantum.occupation(eps, mu, temperature / K_B, statistics), color=colour,
                label=label)
    ax.axvline(mu, color="grey", ls=":", lw=1)
    if log_axis:
        ax.set_yscale("log")
        ax.set_ylim(1e-4, 50)
    else:
        ax.set_ylim(0, 3)
    ax.set_xlabel("eps (energy unit)")
    ax.set_ylabel("<n>")
    ax.legend()
    plt.tight_layout()
    plt.show()
    for name, n, t, m in (("helium gas, 0 C, 1 atm", 101325 / (K_B * 273.15), 273.15, M_HE),
                          ("liquid helium, 2.2 K", N_HE, 2.2, M_HE),
                          ("electrons in copper, 300 K", N_CU, 300.0, M_E)):
        print(f"n lambda^3, {name}: {quantum.degeneracy_parameter(n, t, m):.2e}")


widgets.interact_manual(
    occupations,
    temperature=widgets.FloatLogSlider(value=1.0, base=10, min=-1.5, max=1.0, step=0.05),
    mu=widgets.FloatSlider(value=-1.0, min=-6.0, max=4.0, step=0.1),
    log_axis=widgets.Checkbox(value=False, description="log axis"),
)

**The condensation explorer.** The condensate fraction of a box of $N$ atoms, counted level by
level, against the ideal curve, with the split between the ground state and the excited states at
the temperature you choose. Things to try: $100$ atoms, then $100\,000$; $T = T_c$ for each.

In [ ]:
def condensation(n_atoms=10_000, t_over_tc=0.5):
    t = np.linspace(0.0, 1.5, 76)
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2.5, 1]})
    left.plot(t, quantum.condensate_fraction(t, 1.0), color="k", lw=2, label="ideal")
    left.plot(t, quantum.finite_box_condensate_fraction(n_atoms, t), "--", color="#2563eb",
              label=f"box of {n_atoms:,} atoms")
    here = float(quantum.finite_box_condensate_fraction(n_atoms, t_over_tc))
    left.plot([t_over_tc], [here], "o", color="#d97706", ms=9)
    left.set_xlabel("T / T_c")
    left.set_ylabel("N_0 / N")
    left.legend()
    right.bar(["ground", "excited"], [here, 1 - here], color=["#1e3a8a", "#93c5fd"])
    right.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
    print(f"N0/N = {here:.4f} in the box; ideal {quantum.condensate_fraction(t_over_tc, 1.0):.4f}")


widgets.interact_manual(
    condensation,
    n_atoms=widgets.SelectionSlider(options=[100, 1000, 10_000, 100_000], value=10_000),
    t_over_tc=widgets.FloatSlider(value=0.5, min=0.0, max=1.5, step=0.01),
)

**The Fermi-sea explorer.** The occupied part of the density of states, $g(\varepsilon)\,n_{\mathrm{FD}}$,
under $g(\varepsilon)$, for electrons at copper's density, with the band $\mu \pm \kB T$ that can still
take up heat. Things to try: $T/T_F = 1$, $0.1$ and $0.01$; read off the heat capacity each time.

In [ ]:
def fermi_sea(t_over_tf=0.1):
    eps = np.linspace(0.0, 2.0, 600)  # in units of eps_F
    gas = quantum.ideal_quantum_gas(N_CU, t_over_tf * T_F, M_E, "fd", 2)
    mu = gas.mu / (K_B * T_F)
    density = 1.5 * np.sqrt(eps)  # g(eps) eps_F / N
    filled = density * quantum.occupation(eps, mu, t_over_tf / K_B, "fd")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(eps, density, color="grey")
    ax.fill_between(eps, filled, color="#93c5fd")
    ax.axvspan(max(mu - t_over_tf, 0.0), mu + t_over_tf, color="#d97706", alpha=0.25)
    ax.axvline(1.0, color="k", ls="--", lw=1)
    ax.set_xlabel("eps / eps_F")
    ax.set_ylabel("g(eps) eps_F / N")
    plt.tight_layout()
    plt.show()
    print(f"mu / eps_F = {mu:.4f};  C / (N k_B) = {gas.heat_capacity_per_particle / K_B:.4f} "
          f"(Sommerfeld (pi^2/2) T/T_F = {np.pi**2 / 2 * t_over_tf:.4f}; classical 1.5)")


widgets.interact_manual(
    fermi_sea,
    t_over_tf=widgets.FloatLogSlider(value=0.1, base=10, min=-2, max=0.5, step=0.05),
)

**The chemical-potential tracker.** Part 5's three curves, with the three gases at the temperature
you choose marked on them. Things to try: just above and just below $T_c$; $T = 1.9\,T_c$, where the
classical $\mu$ is zero; $T = 30\,T_c$, where the three agree.

In [ ]:
def mu_tracker(t_over_tc=1.5):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for statistics, colour in (("be", "#2563eb"), ("mb", "#111827"), ("fd", "#dc2626")):
        ax.plot(ratios, mu_curves[statistics], color=colour)
        here = quantum.solve_mu(N_HE, t_over_tc * T_C, M_HE, statistics) / (K_B * T_C)
        ax.plot([t_over_tc], [here], "o", color=colour, ms=8)
        print(f"{statistics}: mu = {here:+.4f} k_B T_c")
    ax.axhline(0.0, color="grey", lw=1)
    ax.set_xscale("log")
    ax.set_ylim(-20, 4)
    ax.set_xlabel("T / T_c")
    ax.set_ylabel("mu / k_B T_c")
    plt.tight_layout()
    plt.show()
    print(f"n lambda^3 = {quantum.degeneracy_parameter(N_HE, t_over_tc * T_C, M_HE):.4f}")


widgets.interact_manual(
    mu_tracker,
    t_over_tc=widgets.FloatLogSlider(value=1.5, base=10, min=-0.5, max=1.45, step=0.01),
)

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "17-quantum-gases.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. Part 2 found bosons together more often than classical particles, with no force between them.
   Explain where the extra probability comes from.
3. Your measurement of silver's $\gamma$ quotes an uncertainty. List every assumption it rests on
   that the uncertainty does *not* include.
4. Below $T_c$ the pressure of a Bose gas does not depend on its density. Explain why, in terms of
   what the condensed atoms are doing.

**Your answers:**

1.
2.
3.
4.